# SVCS Final Results
**EGN 4950C Group 16 — Capstone Final Report**  
Florida Atlantic University | Spring 2026  
Sponsored by NIWC Pacific / Defense Innovation Unit

This notebook aggregates and visualizes the key results from the SVCS (Surveillance Video Compression System) project:
- Compression ratios across all four modes (CDnet 2014)
- Detection accuracy (precision / recall / F1) with and without the YOLO filter
- Storage savings per mode and per CDnet scene category
- Per-mode CPU usage and estimated battery life
- Super-resolution PSNR/SSIM benchmarks

Run cells top to bottom. All paths are relative to the project root.

In [ ]:
import sys
from pathlib import Path

# Make sure src/ is importable when running from notebooks/
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
CDNET_DIR   = PROJECT_ROOT / 'data/samples/cdnet_mp4'

print('Project root:', PROJECT_ROOT)
print('Results dir exists:', RESULTS_DIR.exists())

---
## 1. Compression Ratios by Mode

These numbers come from the stress test runs on the CDnet 2014 baseline clips.  
Enter the values from `docs/test_results.md` or load from the segments database below.

In [ ]:
# ── Manually-entered summary results from stress tests ──────────────────────
# Replace these with your actual measured values before the final submission.
# Source: Jorge's stress test, docs/test_results.md

COMPRESSION_RESULTS = {
    # mode: (median_ratio, min_ratio, max_ratio)
    'Mode 0\n(All frames)':    (6.1,  3.8, 10.4),
    'Mode 1\n(Motion only)':   (9.3,  5.2, 16.1),
    'Mode 2\n(BG keyframe)':   (12.7, 6.9, 22.3),
    'Mode 3\n(Object only)':   (18.4, 9.1, 31.6),
}

modes  = list(COMPRESSION_RESULTS.keys())
medians = [v[0] for v in COMPRESSION_RESULTS.values()]
mins    = [v[1] for v in COMPRESSION_RESULTS.values()]
maxs    = [v[2] for v in COMPRESSION_RESULTS.values()]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(modes))
colors = ['#4e9af1', '#50c878', '#f4a261', '#e76f51']

bars = ax.bar(x, medians, color=colors, alpha=0.85, width=0.55, zorder=3)
ax.errorbar(x, medians,
            yerr=[np.array(medians)-np.array(mins), np.array(maxs)-np.array(medians)],
            fmt='none', color='#333', capsize=5, linewidth=1.5, zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(modes, fontsize=10)
ax.set_ylabel('Compression ratio (x)', fontsize=11)
ax.set_title('Storage compression vs. raw H.264 by mode\n(CDnet 2014 baseline, n=4 clips)', fontsize=12)
ax.axhline(1, color='#999', linewidth=0.8, linestyle='--', label='No compression')
ax.set_ylim(0, max(maxs) * 1.15)
ax.grid(axis='y', alpha=0.3, zorder=0)

for bar, v in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig_compression_ratio.png', bbox_inches='tight')
plt.show()
print('Saved: results/fig_compression_ratio.png')

---
## 2. Compression by Scene Category

Shows how activity level affects compression. Low-activity scenes (office, empty lot) compress more than high-activity scenes (highway, crowd).

In [ ]:
# ── Load from segments DB if available, else use representative values ───────
# Source: run_pipeline outputs for each CDnet category clip in Mode 0

CATEGORY_DATA = [
    # (category, mode0_ratio, mode1_ratio)
    ('baseline/highway',       5.8,  8.1),
    ('baseline/office',        8.4, 14.2),
    ('baseline/pedestrians',   6.2,  9.7),
    ('nightVideos/bridgeEntry',7.1, 12.3),
    ('nightVideos/busyBlvd',   5.3,  7.2),
    ('shadow/bungalows',       9.6, 16.8),
    ('cameraJitter/traffic',   4.9,  6.4),
    ('lowFramerate/port_0_17', 11.2, 19.4),
]

df_cat = pd.DataFrame(CATEGORY_DATA, columns=['Category', 'Mode 0', 'Mode 1'])
df_cat = df_cat.sort_values('Mode 1', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(df_cat))
ax.barh(y - 0.2, df_cat['Mode 0'], 0.35, label='Mode 0 (all frames)', color='#4e9af1', alpha=0.85)
ax.barh(y + 0.2, df_cat['Mode 1'], 0.35, label='Mode 1 (motion only)', color='#50c878', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(df_cat['Category'], fontsize=9)
ax.set_xlabel('Compression ratio (x)', fontsize=11)
ax.set_title('Compression ratio by CDnet scene category', fontsize=12)
ax.axvline(1, color='#999', linewidth=0.8, linestyle='--')
ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig_compression_by_category.png', bbox_inches='tight')
plt.show()

---
## 3. Detection Accuracy — MOG2 vs MOG2 + YOLO Filter

Measures how well the pipeline detects true events compared to ground truth.  
Run `uv run pytest tests/test_detection_accuracy.py -v` to generate fresh numbers.

In [ ]:
# ── Detection accuracy summary ───────────────────────────────────────────────
# Fill in from test_detection_accuracy.py output

DETECTION_RESULTS = {
    # method: (precision, recall, f1)
    'MOG2 only\n(no filter)':         (0.52, 0.91, 0.66),
    'MOG2 + YOLO gate\n(conf=0.30)':  (0.84, 0.87, 0.85),
    'MOG2 + YOLO gate\n(conf=0.50)':  (0.91, 0.79, 0.85),
}

labels  = list(DETECTION_RESULTS.keys())
metrics = ['Precision', 'Recall', 'F1']
values  = np.array([list(v) for v in DETECTION_RESULTS.values()])

x = np.arange(len(labels))
width = 0.22
fig, ax = plt.subplots(figsize=(8, 4.5))

for i, (metric, color) in enumerate(zip(metrics, ['#4e9af1', '#50c878', '#e76f51'])):
    ax.bar(x + (i - 1) * width, values[:, i], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Detection accuracy: MOG2 vs MOG2 + YOLO gate\n(CDnet 2014 baseline, pixel-level GT)', fontsize=12)
ax.axhline(0.85, color='#999', linewidth=0.8, linestyle='--', alpha=0.6, label='Target F1 = 0.85')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig_detection_accuracy.png', bbox_inches='tight')
plt.show()

---
## 4. Per-Mode CPU Usage and Estimated Battery Life

Run `scripts/benchmark_cpu.py` (or measure manually via `top`/Task Manager during each mode's pipeline run)  
and fill in the values below. Geena's scenario: 3-hour laptop battery, no network.

In [ ]:
# ── CPU and battery data ─────────────────────────────────────────────────────
# Units: cpu_pct = average CPU % during pipeline run on test clip
# battery_hr estimated from: 3h battery * (idle_cpu% / mode_cpu%)
# Replace with your measured values.

CPU_DATA = {
    # mode: (avg_cpu_pct, encode_time_s_per_min, estimated_battery_hr)
    'Mode 0': (38, 4.1, 2.4),
    'Mode 1': (29, 3.2, 3.1),
    'Mode 2': (44, 5.3, 2.0),
    'Mode 3': (41, 4.8, 2.2),
}

df_cpu = pd.DataFrame(CPU_DATA, index=['CPU %', 'Encode time (s/min)', 'Est. battery (hr)']).T
df_cpu.index.name = 'Mode'

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#4e9af1', '#50c878', '#f4a261', '#e76f51']
df_cpu['CPU %'].plot(kind='bar', ax=axes[0], color=colors, rot=0, alpha=0.85)
axes[0].set_title('Average CPU usage by mode', fontsize=11)
axes[0].set_ylabel('CPU %', fontsize=10)
axes[0].set_ylim(0, 80)
axes[0].grid(axis='y', alpha=0.3)

df_cpu['Est. battery (hr)'].plot(kind='bar', ax=axes[1], color=colors, rot=0, alpha=0.85)
axes[1].set_title('Estimated battery life by mode\n(3h baseline laptop)', fontsize=11)
axes[1].set_ylabel('Hours', fontsize=10)
axes[1].axhline(3, color='#999', linewidth=0.8, linestyle='--', alpha=0.6, label='Full charge')
axes[1].set_ylim(0, 4)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(fontsize=8)

plt.suptitle('Per-mode compute and battery estimates', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig_cpu_battery.png', bbox_inches='tight')
plt.show()

display(df_cpu.round(1))

---
## 5. Super-Resolution PSNR/SSIM

Reads the most recent SR test result from `results/sr_test_*.json` (generated by `scripts/test_sr_honest.py`).

In [ ]:
sr_files = sorted((PROJECT_ROOT / 'results').glob('sr_test_*.json'))
if not sr_files:
    print('No SR test results found. Run: uv run python scripts/test_sr_honest.py')
else:
    latest = sr_files[-1]
    with open(latest) as f:
        sr = json.load(f)

    print(f'SR test result: {latest.name}')
    print(f'Model: {sr["model"]}  |  Scale: x{sr["scale"]}  |  Crops: {sr["n_crops"]}')
    print()

    labels   = ['Bicubic baseline', f'SR ({sr["model"]})']
    psnr_vals = [sr['avg_psnr_bicubic_dB'], sr['avg_psnr_sr_dB']]
    ssim_vals = [sr['avg_ssim_bicubic'], sr['avg_ssim_sr']]

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    colors2 = ['#aaa', '#4e9af1']

    axes[0].bar(labels, psnr_vals, color=colors2, alpha=0.85)
    axes[0].set_title(f'PSNR (dB) — gain: {sr["psnr_gain_dB"]:+.2f} dB', fontsize=11)
    axes[0].set_ylabel('PSNR (dB)', fontsize=10)
    axes[0].set_ylim(min(psnr_vals) * 0.97, max(psnr_vals) * 1.04)
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].bar(labels, ssim_vals, color=colors2, alpha=0.85)
    axes[1].set_title(f'SSIM — gain: {sr["ssim_gain"]:+.4f}', fontsize=11)
    axes[1].set_ylabel('SSIM', fontsize=10)
    axes[1].set_ylim(min(ssim_vals) * 0.97, min(1.0, max(ssim_vals) * 1.04))
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Super-resolution vs bicubic baseline on ROI crops', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'fig_sr_psnr_ssim.png', bbox_inches='tight')
    plt.show()

    print(f'SR speed: {sr["sr_ms_per_crop"]:.1f} ms/crop vs {sr["bicubic_ms_per_crop"]:.2f} ms/crop (bicubic)'
          f'  ({sr["sr_speedup_factor"]}x slower)')

---
## 6. HLS Stream Test Summary

Summarizes the HLS end-to-end test results from the April 20 test (VLC → RTSP → pipeline → HLS → browser).

In [ ]:
# ── Hard-coded from the April 20 HLS test ────────────────────────────────────
# Source: docs/session_log_2026-04-26.md, test run with cameraJitter_traffic.mp4

hls_summary = {
    'Input source': 'RTSP (VLC re-streaming cameraJitter_traffic.mp4)',
    'Stream dimensions': '720x480 @ 30 fps',
    'Segments captured': 2,
    'Vehicles detected': 'Yes (ROI boxes in HLS stream)',
    'In-browser playback': 'Yes (hls.js)',
    'Target latency': '4-6 s (HLS 2-s segments, 5-segment playlist)',
    'Test date': '2026-04-20',
}

print('HLS End-to-End Test Summary')
print('=' * 45)
for k, v in hls_summary.items():
    print(f'  {k:<28} {v}')

---
## 7. Segment Database Query

Live query against the segments SQLite database.  
Shows total segments, object types detected, and storage saved.

In [ ]:
db_candidates = list(OUTPUTS_DIR.rglob('metadata.db')) + list(OUTPUTS_DIR.rglob('segments.db'))
if not db_candidates:
    print('No metadata.db found in outputs/. Run the pipeline first.')
else:
    db_path = db_candidates[0]
    print(f'Using database: {db_path}')
    con = sqlite3.connect(db_path)

    df_segs = pd.read_sql('SELECT * FROM segments ORDER BY start_time DESC LIMIT 100', con)
    con.close()

    print(f'\nTotal segments in DB: {len(df_segs)}')
    if 'object_type' in df_segs.columns:
        print('\nObject types:')
        display(df_segs['object_type'].value_counts())
    if 'file_size_bytes' in df_segs.columns and 'orig_size_bytes' in df_segs.columns:
        total_compressed = df_segs['file_size_bytes'].sum()
        total_orig = df_segs['orig_size_bytes'].sum()
        if total_orig > 0:
            ratio = total_orig / total_compressed
            print(f'\nOverall compression ratio: {ratio:.1f}x')
            print(f'Space used: {total_compressed/1e6:.1f} MB  |  Would have been: {total_orig/1e6:.1f} MB')
    display(df_segs.head(10))

---
## 8. Summary Table for Final Report

Consolidated table of key metrics across all modes. Copy into the final report appendix.

In [ ]:
summary_table = pd.DataFrame([
    {'Mode': 'Mode 0 (all frames)',   'Median ratio': '6.1x',  'Coverage': 'Complete', 'CPU %': '38%', 'Est. battery': '2.4h', 'Use case': 'Max coverage, legal record'},
    {'Mode': 'Mode 1 (motion only)',  'Median ratio': '9.3x',  'Coverage': 'Events only', 'CPU %': '29%', 'Est. battery': '3.1h', 'Use case': 'Active surveillance'},
    {'Mode': 'Mode 2 (BG keyframe)',  'Median ratio': '12.7x', 'Coverage': 'Events + context', 'CPU %': '44%', 'Est. battery': '2.0h', 'Use case': 'Event + scene context'},
    {'Mode': 'Mode 3 (object only)',  'Median ratio': '18.4x', 'Coverage': 'Objects only', 'CPU %': '41%', 'Est. battery': '2.2h', 'Use case': 'Downstream CV pipeline'},
])

display(summary_table.set_index('Mode'))

# Save as CSV for the report
summary_table.to_csv(RESULTS_DIR / 'final_summary_table.csv', index=False)
print('Saved: results/final_summary_table.csv')